Cell 1

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_log_error

BASE_DIR = Path(r"C:\Users\NickK\OneDrive\Υπολογιστής\github\kaggle_store_sales_forecasting")
ADVANCED_DIR = BASE_DIR / "data" / "advanced"

Cell 2

In [2]:
train = pd.read_csv(ADVANCED_DIR / "train_expanded_ewma.csv")
train["date"] = pd.to_datetime(train["date"])

print(train.shape)

(3000888, 53)


Cell 3

In [3]:
target = "sales"

features = [
    "store_nbr", "family", "onpromotion",
    "year", "month", "day", "quarter", "week", "day_of_week", "is_weekend",
    "city", "state", "type", "cluster",
    "dcoilwtico", "is_payday", "is_national_holiday", "earthquake_period",

    "sales_lag_1", "sales_lag_2", "sales_lag_3",
    "sales_lag_7", "sales_lag_14", "sales_lag_21", "sales_lag_28", "sales_lag_56",

    "rolling_mean_3", "rolling_mean_7", "rolling_mean_14", "rolling_mean_28", "rolling_mean_56",
    "rolling_median_7", "rolling_median_14", "rolling_median_28",

    "rolling_min_7", "rolling_min_14", "rolling_min_28",
    "rolling_max_7", "rolling_max_14", "rolling_max_28",
    "rolling_std_7", "rolling_std_14", "rolling_std_28", "rolling_std_56",

    "promo_rolling_mean_7", "promo_rolling_mean_14", "promo_rolling_mean_28",

    "ewma_03", "ewma_05", "ewma_07"
]

X = train[features]
y = np.log1p(train[target])

print(X.shape)

(3000888, 50)


Cell 4

In [7]:
X = pd.get_dummies(X, drop_first=True)

# Clean column names for LightGBM
X.columns = (
    X.columns
    .str.replace(r"[^A-Za-z0-9_]", "_", regex=True)
)

print(X.shape)

(3000888, 118)


Cell 5

In [8]:
split_date = "2017-01-01"

train_mask = train["date"] < split_date
valid_mask = train["date"] >= split_date

X_train = X.loc[train_mask]
X_valid = X.loc[valid_mask]

y_train = y.loc[train_mask]
y_valid_original = train.loc[valid_mask, target]

print(X_train.shape)
print(X_valid.shape)

(2596374, 118)
(404514, 118)


Cell 6

In [9]:
lgbm_model = LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=-1,
    num_leaves=128,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="regression",
    random_state=42,
    n_jobs=-1
)

lgbm_model.fit(X_train, y_train)

print("LightGBM model trained.")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.167417 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 8949
[LightGBM] [Info] Number of data points in the train set: 2596374, number of used features: 118
[LightGBM] [Info] Start training from score 2.825141
LightGBM model trained.


Cell 7

In [10]:
valid_log_preds = lgbm_model.predict(X_valid)

valid_preds = np.expm1(valid_log_preds)
valid_preds = np.maximum(valid_preds, 0)

rmsle = np.sqrt(
    mean_squared_log_error(
        y_valid_original,
        valid_preds
    )
)

print("Validation RMSLE:", rmsle)

Validation RMSLE: 0.45013975952976326
